# UK Purchasing Power Analysis 2019–2024

## Project Brief

**Business Question:** How has the relationship between wages and inflation changed 
between 2019 and 2024, and what does that mean for the financial reality of UK workers?

**Audience:** Policy analysts, personal finance businesses, and economic researchers 
who need to understand the real impact of the cost of living crisis beyond headline 
inflation figures.

**Data Sources:** ONS CPIH Inflation Index and ONS Average Weekly Earnings (real terms)

**Tools:** Python, Pandas, SQLite, Matplotlib

**Success Metric:** A clear narrative that goes beyond reporting what happened 
to explaining what it means and what should happen next.

In [1]:
import pandas as pd
import sqlite3


cpih = pd.read_csv('CPIH.csv')


wages = pd.read_csv('WAGES.csv', skiprows=8, encoding='latin1')


print("CPIH shape:", cpih.shape)
print(cpih.head())
print("\nWages shape:", wages.shape)
print(wages.head())

CPIH shape: (55510, 7)
    v4_0  mmm-yy  ... cpih1dim1aggid                              Aggregate
0  136.5  Nov-25  ...          CP042     04.2 Owner Occupiers Housing Costs
1  174.6  Nov-25  ...          CP045  04.5 Electricity, gas and other fuels
2  140.0  Nov-25  ...          CP062              06.2 Out-patient services
3  160.2  Nov-25  ...          CP063                 06.3 Hospital services
4  124.2  Nov-25  ...          CP071              07.1 Purchase of vehicles

[5 rows x 7 columns]

Wages shape: (317, 15)
  Unnamed: 0  Unnamed: 1  Unnamed: 2  ...  Unnamed: 12  Unnamed: 13  Unnamed: 14
0     Jan 00       424.0        87.8  ...          NaN          NaN          NaN
1     Feb 00       414.0        85.8  ...          NaN          NaN          NaN
2     Mar 00       429.0        88.9  ...          NaN          NaN          NaN
3     Apr 00       426.0        88.3  ...          NaN          NaN          NaN
4     May 00       430.0        89.1  ...          NaN          NaN   

In [2]:

cpih_clean = cpih[cpih['cpih1dim1aggid'] == 'CP00'][['mmm-yy', 'v4_0']].copy()

cpih_clean.columns = ['date', 'cpih_index']

cpih_clean['date'] = pd.to_datetime(cpih_clean['date'], format='%b-%y')

cpih_clean = cpih_clean[
    (cpih_clean['date'] >= '2019-01-01') & 
    (cpih_clean['date'] <= '2024-12-31')
].sort_values('date').reset_index(drop=True)

print("CPIH cleaned shape:", cpih_clean.shape)
print(cpih_clean.head(10))

CPIH cleaned shape: (72, 2)
        date  cpih_index
0 2019-01-01       106.4
1 2019-02-01       106.8
2 2019-03-01       107.0
3 2019-04-01       107.6
4 2019-05-01       107.9
5 2019-06-01       107.9
6 2019-07-01       108.0
7 2019-08-01       108.3
8 2019-09-01       108.4
9 2019-10-01       108.3


In [3]:

wages_clean = wages.iloc[:313, [0, 1, 5]].copy()

wages_clean.columns = ['date', 'real_awe_total', 'real_awe_regular']


wages_clean['date'] = wages_clean['date'].astype(str).str.replace('-', ' ')

wages_clean['date'] = pd.to_datetime(wages_clean['date'], format='%b %y')

wages_clean = wages_clean[
    (wages_clean['date'] >= '2019-01-01') &
    (wages_clean['date'] <= '2024-12-31')
].sort_values('date').reset_index(drop=True)

print("Wages cleaned shape:", wages_clean.shape)
print(wages_clean.head(10))

Wages cleaned shape: (72, 3)
        date  real_awe_total  real_awe_regular
0 2019-01-01           494.0             467.0
1 2019-02-01           495.0             465.0
2 2019-03-01           496.0             467.0
3 2019-04-01           500.0             468.0
4 2019-05-01           501.0             469.0
5 2019-06-01           502.0             471.0
6 2019-07-01           503.0             470.0
7 2019-08-01           502.0             471.0
8 2019-09-01           503.0             470.0
9 2019-10-01           502.0             472.0


In [4]:

combined = pd.merge(cpih_clean, wages_clean, on='date', how='inner')


base_cpih = combined.loc[combined['date'] == '2019-01-01', 'cpih_index'].values[0]
base_wage = combined.loc[combined['date'] == '2019-01-01', 'real_awe_total'].values[0]

combined['purchasing_power_index'] = (combined['real_awe_total'] / base_wage) * 100
combined['inflation_index'] = (combined['cpih_index'] / base_cpih) * 100

print("Combined shape:", combined.shape)
print(combined.head(10))

Combined shape: (72, 6)
        date  cpih_index  ...  purchasing_power_index  inflation_index
0 2019-01-01       106.4  ...              100.000000       100.000000
1 2019-02-01       106.8  ...              100.202429       100.375940
2 2019-03-01       107.0  ...              100.404858       100.563910
3 2019-04-01       107.6  ...              101.214575       101.127820
4 2019-05-01       107.9  ...              101.417004       101.409774
5 2019-06-01       107.9  ...              101.619433       101.409774
6 2019-07-01       108.0  ...              101.821862       101.503759
7 2019-08-01       108.3  ...              101.619433       101.785714
8 2019-09-01       108.4  ...              101.821862       101.879699
9 2019-10-01       108.3  ...              101.619433       101.785714

[10 rows x 6 columns]


# Act 1: Setting the Scene
## Building the Dataset

Both datasets have been cleaned, merged on date, and indexed to January 2019 = 100. 
This allows us to compare wages and inflation on the same scale and measure the 
growing gap between them over time.

In [5]:


conn = sqlite3.connect(':memory:')

combined.to_sql('purchasing_power', conn, if_exists='replace', index=False)


result = pd.read_sql_query("SELECT * FROM purchasing_power LIMIT 5", conn)
print(result)

                  date  cpih_index  ...  purchasing_power_index  inflation_index
0  2019-01-01 00:00:00       106.4  ...              100.000000       100.000000
1  2019-02-01 00:00:00       106.8  ...              100.202429       100.375940
2  2019-03-01 00:00:00       107.0  ...              100.404858       100.563910
3  2019-04-01 00:00:00       107.6  ...              101.214575       101.127820
4  2019-05-01 00:00:00       107.9  ...              101.417004       101.409774

[5 rows x 6 columns]


In [6]:
q1 = pd.read_sql_query("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(cpih_index), 2) AS avg_cpih,
        ROUND(AVG(inflation_index), 2) AS avg_inflation_index,
        ROUND(MAX(inflation_index) - MIN(inflation_index), 2) AS inflation_rise_within_year
    FROM purchasing_power
    GROUP BY year
    ORDER BY year
""", conn)

print(q1)

   year  avg_cpih  avg_inflation_index  inflation_rise_within_year
0  2019    107.80               101.32                        1.97
1  2020    108.87               102.32                        1.03
2  2021    111.61               104.89                        5.06
3  2022    120.45               113.20                       10.06
4  2023    128.63               120.90                        5.36
5  2024    132.84               124.85                        4.79


In [7]:
q2 = pd.read_sql_query("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(inflation_index), 2) AS avg_inflation_index,
        ROUND(AVG(purchasing_power_index), 2) AS avg_purchasing_power,
        ROUND(AVG(purchasing_power_index) - AVG(inflation_index), 2) AS gap
    FROM purchasing_power
    GROUP BY year
    ORDER BY year
""", conn)

print(q2)

   year  avg_inflation_index  avg_purchasing_power    gap
0  2019               101.32                101.16  -0.15
1  2020               102.32                102.07  -0.24
2  2021               104.89                105.41   0.52
3  2022               113.20                102.72 -10.49
4  2023               120.90                102.41 -18.48
5  2024               124.85                105.16 -19.69


## Insight 1: Workers Never Kept Up

In every year except 2021, inflation outpaced wages. The gap was negligible in 
2019 and 2020, but exploded to -10.49 index points in 2022 and widened further 
to -19.69 by 2024. By the end of the period, prices were nearly 25% higher than 
January 2019 while purchasing power had risen only 5%. Workers were running to 
stand still and losing ground every year.

**So what:** A personal finance business should not be messaging customers about 
budgeting as if the problem is overspending. The data shows the problem is 
structural. Wages simply did not keep pace.

**Now what:** Products and messaging should shift toward financial recovery tools, 
not just budgeting. The customer need in 2024 is rebuilding, not restricting.

In [8]:
q3 = pd.read_sql_query("""
    WITH monthly_gap AS (
        SELECT 
            date,
            ROUND(inflation_index, 2) AS inflation_index,
            ROUND(purchasing_power_index, 2) AS purchasing_power_index,
            ROUND(purchasing_power_index - inflation_index, 2) AS gap,
            LAG(ROUND(purchasing_power_index - inflation_index, 2)) 
                OVER (ORDER BY date) AS prev_gap
        FROM purchasing_power
    )
    SELECT 
        date,
        inflation_index,
        purchasing_power_index,
        gap,
        prev_gap
    FROM monthly_gap
    WHERE gap < 0 AND (prev_gap >= 0 OR prev_gap IS NULL)
    ORDER BY date
""", conn)

print(q3)

                  date  inflation_index  purchasing_power_index   gap  prev_gap
0  2019-02-01 00:00:00           100.38                  100.20 -0.17      0.00
1  2019-08-01 00:00:00           101.79                  101.62 -0.17      0.32
2  2021-09-01 00:00:00           105.64                  105.47 -0.17      0.31


## Insight 2: The Squeeze Started Earlier Than the Headlines Suggested

The data shows wages began falling behind inflation as early as September 2021, 
a full year before the cost of living crisis dominated news coverage. The public 
narrative caught up to the data late. Workers were already losing ground before 
anyone was calling it a crisis.

**So what:** Reactive policy and business responses in 2022 were already a year 
behind the reality workers were experiencing.

**Now what:** Analysts and policymakers should monitor the purchasing power gap 
in real time rather than waiting for inflation headlines to signal a problem.

In [9]:
q4 = pd.read_sql_query("""
    SELECT 
        date,
        ROUND(inflation_index, 2) AS inflation_index,
        ROUND(purchasing_power_index, 2) AS purchasing_power_index,
        ROUND(purchasing_power_index - inflation_index, 2) AS gap
    FROM purchasing_power
    ORDER BY gap ASC
    LIMIT 5
""", conn)

print(q4)

                  date  inflation_index  purchasing_power_index    gap
0  2024-11-01 00:00:00           126.50                  105.87 -20.63
1  2024-12-01 00:00:00           126.97                  106.68 -20.29
2  2024-10-01 00:00:00           126.22                  106.07 -20.15
3  2024-08-01 00:00:00           125.38                  105.26 -20.11
4  2024-06-01 00:00:00           125.00                  105.06 -19.94


## Insight 3: 2024 Was Actually the Worst Year — Not 2022

This is the most counterintuitive finding in the entire dataset. Despite inflation 
slowing in 2023 and 2024, November 2024 was the single worst month for purchasing 
power, with a gap of -20.63 index points. Prices never fell — they just rose more 
slowly — while wages were still catching up from the 2022 shock.

The average worker in late 2024 could buy roughly 20% less than they could have 
if wages had kept pace with prices since January 2019.

**So what:** The commonly held assumption that the cost of living crisis peaked 
in 2022 is misleading. The accumulated damage to purchasing power continued 
worsening through 2024.

**Now what:** Any financial product or policy aimed at cost of living recovery 
should be designed for a 2024 to 2026 recovery window, not a 2022 peak and 
rapid rebound.

In [10]:
q5 = pd.read_sql_query("""
    WITH monthly_changes AS (
        SELECT
            date,
            ROUND(purchasing_power_index, 2) AS purchasing_power_index,
            ROUND(inflation_index, 2) AS inflation_index,
            ROUND(purchasing_power_index - inflation_index, 2) AS gap,
            ROUND(purchasing_power_index - LAG(purchasing_power_index) 
                OVER (ORDER BY date), 2) AS wage_momentum
        FROM purchasing_power
    )
    SELECT
        strftime('%Y', date) AS year,
        ROUND(AVG(gap), 2) AS avg_gap,
        ROUND(AVG(wage_momentum), 2) AS avg_monthly_wage_growth
    FROM monthly_changes
    GROUP BY year
    ORDER BY year
""", conn)

print(q5)

   year  avg_gap  avg_monthly_wage_growth
0  2019    -0.15                     0.07
1  2020    -0.24                     0.39
2  2021     0.52                     0.07
3  2022   -10.49                    -0.44
4  2023   -18.48                     0.17
5  2024   -19.69                     0.30


## Insight 4: Recovery Is Underway But Incomplete

Wage momentum turned positive in 2023 and accelerated in 2024, suggesting workers 
are beginning to recover lost ground. However the accumulated gap from the 2022 
shock remains so large that full recovery had not occurred by the end of 2024. 
The direction is encouraging but the distance still to travel is significant.

**So what:** This is a recovery story, not a solved problem. The labour market 
is responding but the structural damage from 2022 and 2023 has a long tail.

**Now what:** Financial services businesses targeting UK consumers should 
position for a multi-year recovery cycle rather than assuming the crisis is over.

# Final Recommendations

Based on this analysis of UK purchasing power between 2019 and 2024, 
we present the following recommendations for policy analysts, researchers, 
and financial services businesses.

## 1. Stop treating 2022 as the peak of the crisis
The worst month for purchasing power was November 2024, not 2022. Any strategy 
built around a 2022 peak assumption is working from the wrong baseline.

## 2. Monitor the purchasing power gap in real time
The squeeze started in September 2021, a year before it became a headline story. 
Early detection requires tracking wages against inflation monthly, not waiting 
for CPI announcements.

## 3. Design for recovery, not restriction
Workers in 2024 needed tools to rebuild financial resilience, not generic 
budgeting advice. The data shows a structural squeeze, not a behavioural problem.

## 4. Plan for a multi-year recovery window
Wage momentum is positive but the accumulated gap is large. Financial products, 
policy interventions, and economic messaging should be calibrated to a 2024 to 
2027 recovery timeline.